# Ridge Regression

## Why do we need it?

Polynomial regression can fit curves really well — but the more degrees you add, the more it starts **memorizing the training data** instead of learning the actual pattern. This is called **overfitting**.

```
degree 1  →  underfit   (straight line, misses the curve)
degree 2  →  just right (follows the real pattern)
degree 10 →  overfit    (wiggly mess, memorizes every noise point)
```

Ridge regression fixes overfitting by **penalizing large weights** — it forces the model to keep theta values small and well-behaved.

---

## The core idea

Normal linear regression minimizes just the MSE loss:

```
J(θ) = (1/m) · Σ (ŷᵢ - yᵢ)²
```

Ridge adds a penalty term on top:

```
J(θ) = (1/m) · Σ (ŷᵢ - yᵢ)²  +  α · Σ θⱼ²
         ↑                              ↑
    same MSE loss               penalty for large weights
```

The `α · Σ θⱼ²` term is called **L2 regularization**. It says: "yes minimize the error, but also keep the weights small."

---

## What alpha controls

`α` (alpha) is the regularization strength — the most important hyperparameter in Ridge:

| alpha value | effect |
|---|---|
| α = 0 | no penalty → same as plain linear regression |
| α small (0.01) | light penalty → allows larger weights, fits data closely |
| α large (100) | heavy penalty → forces weights toward zero, simpler model |
| α = ∞ | all weights → zero, model predicts the mean every time |

Think of alpha as a dial between **fitting the data** and **keeping the model simple**.

---

## Why squaring the weights (θⱼ²)?

Two reasons:

1. **sign doesn't matter** — a weight of +5 and -5 are both equally penalized. You want to penalize the *magnitude*, not the direction.
2. **math works out cleanly** — the squared penalty has a smooth gradient, so gradient descent can minimize it easily.

---

## What actually happens to theta

Without Ridge — if two features are correlated (e.g. experience and seniority both mean similar things), the model can set one weight to +1000 and another to -999 and get the same prediction. Those huge weights overfit badly.

With Ridge — the penalty punishes large weights, so the model can't do that trick. Weights stay moderate and the model generalizes better to new data.

```
plain regression  →  θ = [20.1,  847.3,  -831.2,  0.4]   ← unstable, overfit
ridge regression  →  θ = [19.8,    3.1,     4.9,  0.3]   ← stable, generalizes
```

---

## sklearn implementation

```python
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

# plain Ridge
ridge_reg = Ridge(alpha=1.0)
ridge_reg.fit(X, y)
ridge_reg.predict([[8/10]])

# Ridge with polynomial features (most common real use case)
ridge_pipeline = Pipeline([
    ("poly", PolynomialFeatures(degree=10, include_bias=False)),
    ("ridge", Ridge(alpha=1.0))
])
ridge_pipeline.fit(X, y)
ridge_pipeline.predict([[8/10]])
```

The pipeline is important here — polynomial features create many terms (some very large), and Ridge keeps their weights under control.

---

## Ridge vs plain LinearRegression

```python
from sklearn.linear_model import LinearRegression, Ridge

# degree 10 polynomial — will badly overfit without Ridge
poly = PolynomialFeatures(degree=10, include_bias=False)
X_poly = poly.fit_transform(X)

lin = LinearRegression().fit(X_poly, y)   # overfits, huge weights
rdg = Ridge(alpha=1.0).fit(X_poly, y)     # controlled, smaller weights

print(lin.coef_)   # something like [0.3, 841.2, -1203.4, ...]
print(rdg.coef_)   # something like [0.3,   2.1,     1.8, ...]
```

---

## Choosing alpha — cross validation

You should never just guess alpha. Use `RidgeCV` to try multiple values automatically:

```python
from sklearn.linear_model import RidgeCV

ridge_cv = RidgeCV(alphas=[0.001, 0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
ridge_cv.fit(X_poly, y)

print(ridge_cv.alpha_)      # best alpha found
print(ridge_cv.coef_)       # weights with best alpha
```

`cv=5` means 5-fold cross validation — it splits your data 5 ways, trains on 4 parts, tests on 1, rotates, and picks the alpha with the best average validation error.

---

## Ridge vs Lasso

Ridge is not the only regularization method. The other common one is **Lasso (L1 regularization)**:

```
Ridge (L2)  →  penalty = α · Σ θⱼ²     shrinks weights toward zero, never exactly zero
Lasso (L1)  →  penalty = α · Σ |θⱼ|    can shrink weights to exactly zero (feature selection)
```

The key difference:

- **Ridge** — keeps all features, just makes their weights small. Good when all features matter a little.
- **Lasso** — eliminates features entirely by zeroing out their weights. Good when you suspect many features are irrelevant.

In practice Ridge is the safer default. Use Lasso when you have many features and want the model to automatically select the important ones.

---

## Summary

```
problem          overfitting — model memorizes training data, fails on new data
cause            too many features / large weights
Ridge solution   add α · Σ θⱼ² penalty to loss → forces weights to stay small
alpha            controls tradeoff: 0 = no regularization, large = very constrained
use with         polynomial features especially — high degree without Ridge = disaster
tune with        RidgeCV — let cross validation find the best alpha
```
